# 01 · Retrieval medido: la escalera de recall@5

Este notebook ejecuta y mide los cuatro escalones de mejora del retrieval sobre el **golden set oficial** (las 13 preguntas con ancla de texto). Reglas del juego:

- **Medición a nivel de componente**, no del agente entero: conjunto fijo de consultas → búsqueda → ¿el ancla aparece en el top-k? Aísla el retrieval del ruido del modelo y no cuesta ni un céntimo (salvo el escalón D, que reescribe consultas).
- **Un cambio, una medida.** Cada escalón añade exactamente un arreglo sobre el anterior; si la métrica se mueve, sabemos por qué.
- **recall@5** como métrica (K=5 fijado por el proyecto), con **posición del ancla** como diagnóstico de los fallos.
- El ancla es una **frase literal** del informe, no un `chunk_id`: la métrica sobrevive a cualquier re-troceado futuro.

Importante: este notebook **no toca la herramienta del agente**. `search_filings` sigue usando la búsqueda densa con filtros hasta que el baseline esté congelado; la configuración ganadora de aquí se conectará después, como mejora medida.

In [1]:
import json
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "agente").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import pandas as pd

from agente import datos, metricas, retrieval

golden = [json.loads(l)
          for l in (RAIZ / "golden/oficial_20.jsonl").read_text(encoding="utf-8").splitlines()
          if l.strip()]
ancladas = [g for g in golden if g.get("ancla_texto")]
print(f"Golden oficial: {len(golden)} preguntas · con ancla: {len(ancladas)} "
      f"({sum(1 for g in ancladas if g['familia']=='extractiva')} extractivas, "
      f"{sum(1 for g in ancladas if g['familia']=='comparativa')} comparativas)")

datos.cargar_indice()   # calienta índice + codificador (descarga BGE la 1ª vez)
print("Índice y codificador cargados.")

Golden oficial: 20 preguntas · con ancla: 13 (6 extractivas, 7 comparativas)


/Users/eduardogonzalezarroyo/Desktop/Material MIAX/Practicas Entregables/Taller NLP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8025.87it/s]


Índice y codificador cargados.


## Escalón A — denso plano

El punto de partida del día 10: la pregunta **en español, tal cual**, contra todo el índice, sin filtros. Es deliberadamente ingenuo — es lo que hay antes de arreglar nada, y su recall es la línea desde la que se mide todo lo demás.

La expectativa es un recall bajo, y conviene entender por qué antes de verlo: el codificador (`bge-small-en-v1.5`) es **monolingüe en inglés** y el corpus también; una consulta en español cae lejos en el espacio de embeddings aunque pregunte exactamente por lo que el ancla dice. No es código roto — es un desajuste de idioma, y esa distinción decide qué arreglo funciona (el D) y cuáles no pueden funcionar solos (el B y el C).

In [2]:
def escalon_A(item):
    """Pregunta cruda, sin filtros: el retrieval del dia 10."""
    return retrieval.buscar_densa(item["pregunta"], k=metricas.K)

recall_A, detalle_A = metricas.recall_en_k(ancladas, escalon_A)
print(f"A · denso plano           recall@5 = {recall_A:.2f}   "
      f"({sum(detalle_A.values())}/{len(detalle_A)})")
print("  fallos:", sorted(i for i, ok in detalle_A.items() if not ok))

A · denso plano           recall@5 = 0.31   (4/13)
  fallos: ['of-001', 'of-004', 'of-005', 'of-006', 'of-015', 'of-016', 'of-018', 'of-019', 'of-020']


## Escalón B — filtros de metadatos

Mismo buscador, pero restringido al `(ticker, fiscal_year, item)` que la pregunta declara — aquí los aporta el golden; a nivel de agente los aporta el propio modelo como argumentos de la tool, así que la simulación es fiel.

El argumento: **buscar donde hay que buscar no es lo mismo que ordenar bien**, pero elimina de golpe las colisiones entre compañías (el riesgo de IA de Microsoft y el de Meta se parecen mucho más entre sí que cualquiera de los dos a una consulta en español). El filtro no mejora el ranking dentro de la sección correcta; solo garantiza que los 5 puestos del top se gastan en ella.

In [3]:
def escalon_B(item):
    """Pregunta cruda + filtros de metadatos del propio golden."""
    return retrieval.buscar_densa(item["pregunta"], ticker=item["ticker"],
                                  fiscal_year=item["fiscal_year"],
                                  item=item.get("item_esperado"),
                                  k=metricas.K)

recall_B, detalle_B = metricas.recall_en_k(ancladas, escalon_B)
print(f"B · + filtros             recall@5 = {recall_B:.2f}   "
      f"({sum(detalle_B.values())}/{len(detalle_B)})")
print("  fallos:", sorted(i for i, ok in detalle_B.items() if not ok))

B · + filtros             recall@5 = 0.46   (6/13)
  fallos: ['of-001', 'of-004', 'of-005', 'of-015', 'of-016', 'of-018', 'of-019']


## Escalón C — híbrido denso + BM25 (fusión RRF)

Se añade un ranking léxico (BM25) y se fusiona con el denso mediante **Reciprocal Rank Fusion**: `RRF(d) = 1/(60+pos_densa) + 1/(60+pos_bm25)`. Fusión por posiciones y no por suma de puntuaciones, porque una similitud coseno en [-1, 1] y un score BM25 sin escala fija **no son sumables** — las posiciones sí son comparables entre listas. RRF además no tiene pesos que ajustar, y con 13 preguntas de muestra, ajustar pesos sería sobreajuste garantizado.

Expectativa honesta, con el precedente de clase delante: sobre estas mismas 13 consultas **en español**, el híbrido no movió el recall (0.54 → 0.54). Tiene lógica — si la consulta está en otro idioma, el solape léxico con el informe es tan pobre como el semántico. BM25 aporta cuando hay términos exactos que clavar (tickers, nombres de producto); su momento llega **después** de la reescritura, no antes. Si aquí tampoco se mueve, no es un fallo del arreglo: es un resultado, y va al informe como tal.

In [4]:
def escalon_C(item):
    """Hibrido RRF + filtros, consulta aun en espanol."""
    return retrieval.buscar_hibrida(item["pregunta"], ticker=item["ticker"],
                                    fiscal_year=item["fiscal_year"],
                                    item=item.get("item_esperado"),
                                    k=metricas.K)

recall_C, detalle_C = metricas.recall_en_k(ancladas, escalon_C)
print(f"C · + híbrido RRF         recall@5 = {recall_C:.2f}   "
      f"({sum(detalle_C.values())}/{len(detalle_C)})")
print("  fallos:", sorted(i for i, ok in detalle_C.items() if not ok))

C · + híbrido RRF         recall@5 = 0.46   (6/13)
  fallos: ['of-001', 'of-002', 'of-004', 'of-015', 'of-016', 'of-018', 'of-019']


## Escalón D — reescritura de consulta ES→EN

El arreglo que ataca el cuello real: cada pregunta se reescribe **una vez** como consulta de búsqueda en inglés, con vocabulario de 10-K, usando un modelo lite a `temperature=0` (coste: una llamada mínima por consulta distinta, cacheada en proceso). Sobre la consulta reescrita se aplica todo lo anterior (filtros + híbrido): la escalera es acumulativa.

Dos aclaraciones metodológicas que conviene dejar escritas:

1. **La reescritura es pieza del arnés de medición, no de la herramienta.** El agente ya escribe sus consultas en inglés porque el system prompt se lo exige (verificado en el humo de la entrega 2: consultó `"artificial intelligence misuse third parties…"`). Meter la reescritura dentro de `search_filings` duplicaría la traducción y añadiría una llamada de modelo a cada búsqueda del agente.
2. Aquí el BM25 del escalón C **empieza a tener sentido**: con la consulta en inglés, los términos exactos del informe por fin pueden solapar.

In [5]:
import os

if not os.environ.get("OPENROUTER_API_KEY"):
    from getpass import getpass
    os.environ["OPENROUTER_API_KEY"] = getpass("OPENROUTER_API_KEY: ")

reescritas = {it["id"]: retrieval.reescribir(it["pregunta"]) for it in ancladas}

sin_reescribir = [i for i, q in reescritas.items()
                  if q == next(x["pregunta"] for x in ancladas if x["id"] == i)]
if sin_reescribir:
    print(f"AVISO: {len(sin_reescribir)} consultas sin reescribir (¿fallo de "
          f"API?): {sin_reescribir} — el escalón D no es fiable en esas.")

for it in ancladas[:4]:
    print(f"{it['id']}: {reescritas[it['id']]}")
print("…")

def escalon_D(item):
    """Hibrido RRF + filtros sobre la consulta reescrita en ingles."""
    return retrieval.buscar_hibrida(reescritas[item["id"]],
                                    ticker=item["ticker"],
                                    fiscal_year=item["fiscal_year"],
                                    item=item.get("item_esperado"),
                                    k=metricas.K)

recall_D, detalle_D = metricas.recall_en_k(ancladas, escalon_D)
print(f"\nD · + reescritura ES→EN   recall@5 = {recall_D:.2f}   "
      f"({sum(detalle_D.values())}/{len(detalle_D)})")
print("  fallos:", sorted(i for i, ok in detalle_D.items() if not ok))

of-001: China market competition export controls US government restrictions impact on business
of-002: Generative AI security risks internal systems Fiscal Year 2025 risk factors
of-003: Meta Platforms EU US data transfer legal basis GDPR standard contractual clauses adequacy decision FY2025 10-K
of-004: tariff risk trade policy import duties international trade restrictions
…

D · + reescritura ES→EN   recall@5 = 0.77   (10/13)
  fallos: ['of-001', 'of-015', 'of-020']


## La escalera completa

In [6]:
tabla = pd.DataFrame(
    [("A", "denso plano (pregunta ES, sin filtros)", recall_A),
     ("B", "+ filtros de metadatos", recall_B),
     ("C", "+ híbrido RRF", recall_C),
     ("D", "+ reescritura ES→EN", recall_D)],
    columns=["escalón", "configuración", "recall@5"],
).assign(n_ancladas=len(ancladas))
display(tabla)

detalle = pd.DataFrame(
    {"A": detalle_A, "B": detalle_B, "C": detalle_C, "D": detalle_D}
).sort_index()
display(detalle)

(RAIZ / "resultados").mkdir(exist_ok=True)
tabla.to_csv(RAIZ / "resultados/recall_escalera.csv", index=False)
detalle.to_csv(RAIZ / "resultados/recall_detalle.csv")
print("Guardado: resultados/recall_escalera.csv · resultados/recall_detalle.csv")

,escalón,configuración,recall@5,n_ancladas
0,A,"denso plano (pregunta ES, sin filtros)",0.307692,13
1,B,+ filtros de metadatos,0.461538,13
2,C,+ híbrido RRF,0.461538,13
3,D,+ reescritura ES→EN,0.769231,13


,A,B,C,D
of-001,False,False,False,False
of-002,True,True,False,True
of-003,True,True,True,True
of-004,False,False,False,True
of-005,False,False,True,True
of-006,False,True,True,True
of-014,True,True,True,True
of-015,False,False,False,False
of-016,False,False,False,True
of-017,True,True,True,True


Guardado: resultados/recall_escalera.csv · resultados/recall_detalle.csv


## Diagnóstico de los fallos que quedan

Para cada pregunta que el escalón D no recupera en el top-5: **¿en qué puesto del ranking completo está el ancla?** La distinción importa porque decide la siguiente mejora. Un ancla en el puesto 6-15 es un problema de *orden* — candidata a re-ranking con cross-encoder o a subir k (la puerta que la propia S2 deja abierta en la celda 14 con el caso of-001). Un ancla en el puesto 200 es un problema de *representación* — ni reordenar ni subir k la salvan, y habría que mirar el troceado o el embedding.

In [7]:
fallos_D = [it for it in ancladas if not detalle_D[it["id"]]]
if not fallos_D:
    print("Sin fallos en el escalón D sobre estas 13.")
for it in fallos_D:
    ranking = retrieval.buscar_hibrida(reescritas[it["id"]],
                                       ticker=it["ticker"],
                                       fiscal_year=it["fiscal_year"],
                                       item=it.get("item_esperado"),
                                       k=10_000)   # ranking completo del subconjunto filtrado
    pos = metricas.posicion_del_ancla(ranking, it["ancla_texto"])
    print(f"{it['id']}: ancla en el puesto {pos}"
          f"   ·   consulta: {reescritas[it['id']][:60]}…"
          f"   ·   {it['pregunta'][:60]}…")

of-001: ancla en el puesto 9   ·   consulta: China market competition export controls US government restr…   ·   ¿Qué dice NVIDIA en su 10-K de FY2025 sobre la competencia e…
of-015: ancla en el puesto 6   ·   consulta: NVIDIA revenue growth increase fiscal year 2024 to 2025 driv…   ·   ¿Cuánto creció el revenue de NVIDIA entre FY2024 y FY2025, y…
of-020: ancla en el puesto 11   ·   consulta: Meta Platforms net income 2025 compared to 2024 drivers of c…   ·   ¿Cómo varió el beneficio neto de Meta entre 2024 y 2025, y q…


In [11]:
# Comprobación B vs C sobre las consultas REESCRITAS: ¿aporta BM25 algo
# cuando la consulta ya está en inglés, que es como consulta el agente?
try:
    reescritas
except NameError:
    reescritas = {it["id"]: retrieval.reescribir(it["pregunta"]) for it in ancladas}

def densa_reescrita(item):
    return retrieval.buscar_densa(reescritas[item["id"]], ticker=item["ticker"],
                                  fiscal_year=item["fiscal_year"],
                                  item=item.get("item_esperado"), k=metricas.K)

recall_Dp, detalle_Dp = metricas.recall_en_k(ancladas, densa_reescrita)
print(f"densa   + filtros + reescritura   recall@5 = {recall_Dp:.2f}  "
      f"({sum(detalle_Dp.values())}/{len(detalle_Dp)})")
print(f"híbrida + filtros + reescritura   recall@5 = {recall_D:.2f}  "
      f"({sum(detalle_D.values())}/{len(detalle_D)})   (escalón D)")

difieren = sorted(i for i in detalle_D if detalle_D[i] != detalle_Dp[i])
for i in difieren:
    print(f"  {i}: densa={'OK' if detalle_Dp[i] else 'X'}   "
          f"híbrida={'OK' if detalle_D[i] else 'X'}")
if not difieren:
    print("  idénticas pregunta a pregunta: BM25 ni suma ni resta en inglés.")
    
pd.DataFrame({"config": ["densa+filtros+reescritura", "hibrida+filtros+reescritura"],
              "recall@5": [recall_Dp, recall_D]}).assign(n_ancladas=len(ancladas)) \
  .to_csv(RAIZ / "resultados/densa_vs_hibrida_reescritas.csv", index=False)
print("Guardado: resultados/densa_vs_hibrida_reescritas.csv")

densa   + filtros + reescritura   recall@5 = 0.69  (9/13)
híbrida + filtros + reescritura   recall@5 = 0.77  (10/13)   (escalón D)
  of-015: densa=OK   híbrida=X
  of-018: densa=X   híbrida=OK
  of-019: densa=X   híbrida=OK
Guardado: resultados/densa_vs_hibrida_reescritas.csv


## Lectura y decisión

## Lectura y decisión

**Escalera medida (N = 13 anclas del golden oficial):** 0.31 → 0.46 → 0.46 → 0.77, contra la de clase 0.31 → 0.54 → 0.54 → 0.85. El patrón se reproduce: el cuello es el **idioma** (el salto está en la reescritura), no el algoritmo de búsqueda.

**Tres observaciones del detalle por pregunta:**
1. El híbrido sobre consultas en español no fue neutro sino un intercambio: perdió of-002 y ganó of-005 (neto 0). Sin solape léxico, BM25 reordena pero no mejora.
2. La reescritura no es monótona: recuperó 5 preguntas y rompió of-020 (su ancla pasó del top-5 al puesto 11). La calidad de la reescritura es una palanca real, pero afinar su prompt contra estas mismas 13 sería ajustar sobre el set de medida; se deja como está.
3. Los tres fallos del escalón D (of-001, of-015, of-020) tienen el ancla en los puestos **9, 6 y 11**: problema de *orden*, no de representación. Mejora designada para la fase de mejoras: **re-ranking con cross-encoder sobre un top-20**, medido con este mismo arnés.

**Decisión sobre la herramienta** (`search_filings`, al descongelar el baseline). La reescritura queda fuera de la tool: el agente ya consulta en inglés por system prompt (verificado en el humo de la entrega 2). Entre densa e híbrida, la comprobación en régimen inglés da **densa 0.69 (9/13) vs. híbrida 0.77 (10/13)**; difieren of-015 (densa recupera, híbrida pierde) y of-018/of-019 (al revés). BM25 vuelve a intercambiar, pero con la consulta en inglés el neto es **+1**, confirmando la predicción escrita de antemano en el escalón C. Por el criterio prefijado (mayor recall@5 en régimen inglés), **la tool pasará a híbrida+filtros** al descongelar. Dos cautelas: +1 sobre N=13 es el salto mínimo detectable (±0,077), y el ancla de of-015 queda en el puesto 6 — justo el casi-fallo que el re-ranker sobre top-20 está llamado a recuperar, lo que hace ambas mejoras coherentes entre sí.

**Límite:** N = 13 → cada pregunta vale ±0,077 de recall. Las diferencias se leen en saltos, no en décimas.

In [8]:
print("NOTEBOOK 01 COMPLETADO")
print("Ficheros escritos: resultados/recall_escalera.csv · resultados/recall_detalle.csv")
print("Siguiente: evaluadores + evaluar() (notebook 02 llegará con la tabla baseline vs final)")

NOTEBOOK 01 COMPLETADO
Ficheros escritos: resultados/recall_escalera.csv · resultados/recall_detalle.csv
Siguiente: evaluadores + evaluar() (notebook 02 llegará con la tabla baseline vs final)
